# Analyse des systèmes éducatifs
## Recommandation de pays pour une expansion internationale

Ce notebook présente une démarche complète de préparation, d’analyse et de sélection de pays à partir du dataset **EdStats**.

L’objectif est de construire une **base quantitative d’aide à la décision** pour une réflexion d’expansion à l’international, en s’appuyant sur des indicateurs :

- éducatifs ;
- numériques ;
- socio-économiques.

Le notebook a été réorganisé pour être :

- lisible ;
- reproductible ;
- conforme à l’énoncé ;
- exploitable dans JupyterLab comme dans VS Code.

## Plan du notebook

1. Chargement et exploration des fichiers  
2. Identification et suppression des faux pays  
3. Réduction du périmètre des indicateurs  
4. Réduction du périmètre des années  
5. Mesure de la richesse des données  
6. Agrégation à la maille *un pays = une ligne*  
7. Corrélations et suppression des indicateurs redondants  
8. Analyse univariée des indicateurs finaux  
9. Construction d’un score pays  
10. Affichage final des pays recommandés

## Librairies utilisées et rôle de Pandas / NumPy / Matplotlib

Dans ce notebook :

- **Pandas** sert à charger, filtrer, transformer et agréger les tableaux de données (`read_csv`, `groupby`, `pivot_table`, `merge`, `melt`, `isna`, `value_counts`, etc.).
- **NumPy** sert à faire des opérations numériques rapides, notamment pour la normalisation et certains calculs matriciels.
- **Matplotlib** sert à construire des graphiques simples et lisibles : bar charts, histogrammes, boxplots et graphiques finaux.
- **Seaborn** est utilisé en complément pour la heatmap des corrélations.

## Contexte métier et fil directeur du projet

Vous jouez ici le rôle d’un **Data Scientist chez academy**, une start-up EdTech qui souhaite étudier son **expansion à l’international**.

Le notebook suit volontairement la logique pédagogique du projet OpenClassrooms :

1. **explorer** les 5 fichiers ;
2. **fiabiliser** les données ;
3. **retirer les faux pays** ;
4. **réduire le périmètre** des années et des indicateurs ;
5. **agréger** les données à la maille pays ;
6. **analyser les corrélations** ;
7. **construire une recommandation quantitative** de pays cibles.

## Conseils de lecture et de travail

Pour rester proche des recommandations pédagogiques :
- chaque cellule correspond à une tâche précise ;
- les parties sont séparées par des titres Markdown ;
- les hypothèses métier sont expliquées au fur et à mesure ;
- le code privilégie les méthodes Pandas usuelles : `head()`, `shape`, `duplicated()`, `drop_duplicates()`, `isna()`, `value_counts()`, `groupby()`, `pivot_table()`.

## Rôle des bibliothèques utilisées

- **Pandas** : charger, filtrer, nettoyer, agréger et analyser les DataFrames ;
- **NumPy** : manipuler efficacement des listes/arrays numériques, générer des plages d’années avec `np.arange()`, vectoriser certains calculs ;
- **Matplotlib** : tracer des graphiques lisibles pour interpréter les données ;
- **Missingno** : visualiser rapidement les valeurs manquantes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## Localisation automatique des fichiers

Le notebook cherche automatiquement les fichiers EdStats dans plusieurs emplacements possibles :

- le dossier courant ;
- le dossier parent ;
- un sous-dossier `cleaned/` si présent ;
- la racine du projet si Jupyter est lancé depuis `notebooks/`.

Cette approche évite de coder en dur un chemin absolu.

In [ ]:
def trouver_dossier_donnees():
    candidats = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "cleaned",
        Path.cwd().parent / "cleaned",
        Path.cwd().parent.parent,
        Path.cwd().parent.parent / "cleaned",
    ]

    fichiers_attendus = {
        "EdStatsCountry.csv",
        "EdStatsCountry-Series.csv",
        "EdStatsData.csv",
        "EdStatsFootNote.csv",
        "EdStatsSeries.csv",
    }

    for dossier in candidats:
        if dossier.exists():
            contenu = {p.name for p in dossier.iterdir() if p.is_file()}
            if fichiers_attendus.issubset(contenu):
                return dossier

    raise FileNotFoundError(
        "Impossible de trouver les fichiers EdStats. "
        "Placez le notebook dans le projet ou lancez Jupyter depuis la racine du projet."
    )

base_path = trouver_dossier_donnees()
print("Dossier utilisé :", base_path)
print("\nCSV détectés :")
for nom in sorted([p.name for p in base_path.iterdir() if p.is_file() and p.suffix == ".csv"]):
    print("-", nom)

## Chargement des fichiers

Je charge maintenant les cinq fichiers principaux du dataset :

- `Country`
- `Country-Series`
- `Data`
- `FootNote`
- `Series`

In [ ]:
country = pd.read_csv(base_path / "EdStatsCountry.csv")
country_series = pd.read_csv(base_path / "EdStatsCountry-Series.csv")
data = pd.read_csv(base_path / "EdStatsData.csv")
footnote = pd.read_csv(base_path / "EdStatsFootNote.csv")
series = pd.read_csv(base_path / "EdStatsSeries.csv")

print("country        :", country.shape)
print("country_series :", country_series.shape)
print("data           :", data.shape)
print("footnote       :", footnote.shape)
print("series         :", series.shape)

## Ce que représente une ligne dans chaque fichier

Avant de nettoyer les données, il faut comprendre la **maille** de chaque table :

- **Country** : une ligne décrit un **pays** (ou parfois un faux pays / agrégat régional) ;
- **Country-Series** : une ligne décrit une **combinaison pays + série** ;
- **Data** : une ligne décrit une **combinaison pays + indicateur**, avec les années en colonnes ;
- **Series** : une ligne décrit un **indicateur** ;
- **FootNote** : une ligne décrit généralement une **note** associée à un pays, une série et parfois une année.

Cette étape est essentielle, car on ne nettoie pas de la même façon un fichier à la maille *pays* qu’un fichier à la maille *indicateur* ou *pays × indicateur*.

In [ ]:
resume_fichiers = pd.DataFrame(
    [
        ["EdStatsCountry.csv", "1 ligne ≈ 1 pays ou agrégat"],
        ["EdStatsCountry-Series.csv", "1 ligne ≈ 1 combinaison pays + série"],
        ["EdStatsData.csv", "1 ligne ≈ 1 combinaison pays + indicateur ; années en colonnes"],
        ["EdStatsSeries.csv", "1 ligne ≈ 1 indicateur"],
        ["EdStatsFootNote.csv", "1 ligne ≈ 1 note liée à un pays / une série / une année"],
    ],
    columns=["Fichier", "Interprétation d'une ligne"],
)
display(resume_fichiers)

# 1. Exploration initiale des fichiers

Pour chaque table, je vérifie :

- les premières lignes ;
- le nombre de lignes et de colonnes ;
- les doublons ;
- les valeurs manquantes ;
- les colonnes inutilisables ;
- les statistiques descriptives pour les variables numériques ;
- un aperçu des variables catégorielles.

L’idée est de **comprendre la structure** des données avant de commencer le nettoyage.

In [ ]:
def explorer_nettoyer_dataframe(df, nom_fichier):
    print("\n" + "=" * 70)
    print(f"Exploration de : {nom_fichier}")
    print("=" * 70)

    print("\nPremières lignes :")
    display(df.head())

    nb_lignes, nb_colonnes = df.shape
    print(f"\nNombre de lignes   : {nb_lignes}")
    print(f"Nombre de colonnes : {nb_colonnes}")

    nb_doublons = df.duplicated().sum()
    print(f"\nNombre de doublons : {nb_doublons}")

    df = df.drop_duplicates().copy()
    print(f"Nombre de lignes après suppression des doublons : {df.shape[0]}")

    proportion_nan = (df.isna().sum() / len(df)) * 100
    print("\nProportion de valeurs manquantes par colonne :")
    display(proportion_nan.sort_values(ascending=False).to_frame(name="pct_nan"))

    colonnes_inutilisables = proportion_nan[proportion_nan == 100].index.tolist()
    print("\nColonnes inutilisables :", colonnes_inutilisables)

    if colonnes_inutilisables:
        df = df.drop(columns=colonnes_inutilisables)
        print("Colonnes supprimées.")
    else:
        print("Aucune colonne supprimée.")

    colonnes_numeriques = df.select_dtypes(include="number").columns
    if len(colonnes_numeriques) > 0:
        print("\nStatistiques descriptives des colonnes numériques :")
        display(df[colonnes_numeriques].describe())

    colonnes_categorielles = df.select_dtypes(include=["object", "string"]).columns
    if len(colonnes_categorielles) > 0:
        print("\nAperçu des variables catégorielles :")
        for col in colonnes_categorielles[:5]:
            print(f"\nColonne : {col}")
            display(df[col].value_counts(dropna=False).head(10))

    return df

In [ ]:
country = explorer_nettoyer_dataframe(country, "EdStatsCountry.csv")
country_series = explorer_nettoyer_dataframe(country_series, "EdStatsCountry-Series.csv")
data = explorer_nettoyer_dataframe(data, "EdStatsData.csv")
footnote = explorer_nettoyer_dataframe(footnote, "EdStatsFootNote.csv")
series = explorer_nettoyer_dataframe(series, "EdStatsSeries.csv")

## Visualisation rapide des valeurs manquantes

`missingno` permet d’obtenir une vue synthétique des données manquantes.  
Je l’utilise ici de manière légère sur quelques tables principales.

In [ ]:
msno.bar(country)
plt.title("Valeurs manquantes - Country")
plt.show()

msno.bar(data.iloc[:, :20])
plt.title("Aperçu des valeurs manquantes - premières colonnes de Data")
plt.show()

## Pourquoi analyser les valeurs manquantes ?

En analyse exploratoire, les valeurs manquantes ne sont pas un simple détail :
elles conditionnent directement ce qu’il sera **possible** d’exploiter ensuite.

Avec **Pandas**, on les mesure via `isna()` / `isnull()`.  
Avec **Missingno**, on les visualise rapidement.  
Cela permet ensuite de :
- repérer les colonnes inutilisables ;
- comparer les années entre elles ;
- choisir des indicateurs plus robustes.

# 2. Identification et suppression des faux pays

Le fichier `Country` ne contient pas uniquement des pays.  
On y trouve aussi des agrégats régionaux ou économiques comme :

- `World`
- `Arab World`
- `High income`
- `Sub-Saharan Africa`
- `Europe & Central Asia`

Pour fiabiliser l’analyse, je retire ces **faux pays** des tables où cela fait sens :
- `Country`
- `Country-Series`
- `FootNote`
- `Data`

In [ ]:
print(country.columns.tolist())

colonnes_country = [
    col for col in ["Country Code", "Short Name", "Table Name", "Region", "Income Group"]
    if col in country.columns
]
display(country[colonnes_country].head(30))

## Repérage des faux pays

Une règle simple et robuste consiste à considérer comme faux pays les lignes pour lesquelles :

- `Region` est manquante ;
- ou `Income Group` est manquante.

Cette règle capture une grande partie des agrégats non-pays du dataset.

In [ ]:
colonnes_faux_pays = [
    col for col in ["Country Code", "Short Name", "Table Name", "Region", "Income Group"]
    if col in country.columns
]

faux_pays = country[
    country["Region"].isna() | country["Income Group"].isna()
][colonnes_faux_pays].copy()

display(faux_pays.head(30))
print("Nombre de faux pays identifiés :", faux_pays.shape[0])

liste_faux_pays = faux_pays["Country Code"].dropna().unique().tolist()
print("Aperçu des codes supprimés :", liste_faux_pays[:20])

## Méthode 1 — filtrage avec une liste

Ici, j’utilise une liste Python contenant les codes des faux pays.

Avec **Pandas**, l’expression :

```python
df[~df[colonne].isin(liste)]
```

signifie :
- `isin(liste)` : teste si la valeur appartient à la liste ;
- `~` : inverse la condition ;
- on ne garde donc **que les lignes valides**.

In [ ]:
footnote_country_col = "CountryCode" if "CountryCode" in footnote.columns else "Country Code"

country_clean = country[~country["Country Code"].isin(liste_faux_pays)].copy()
country_series_clean = country_series[~country_series["CountryCode"].isin(liste_faux_pays)].copy()
footnote_clean = footnote[~footnote[footnote_country_col].isin(liste_faux_pays)].copy()
data_clean = data[~data["Country Code"].isin(liste_faux_pays)].copy()

print("country        :", country.shape, "->", country_clean.shape)
print("country_series :", country_series.shape, "->", country_series_clean.shape)
print("footnote       :", footnote.shape, "->", footnote_clean.shape)
print("data           :", data.shape, "->", data_clean.shape)

## Méthode 2 — filtrage avec un inner join

Je vérifie le même résultat via un **inner join**.

Avec **Pandas**, `merge(..., how="inner")` garde uniquement les lignes ayant une clé commune dans les deux tables.  
Ici, cela revient à conserver uniquement les pays présents dans `country_clean`.

In [ ]:
pays_valides = country_clean[["Country Code"]].drop_duplicates()

country_series_join = country_series.merge(
    pays_valides,
    left_on="CountryCode",
    right_on="Country Code",
    how="inner"
)

footnote_join = footnote.merge(
    pays_valides,
    left_on=footnote_country_col,
    right_on="Country Code",
    how="inner"
)

data_join = data.merge(
    pays_valides,
    on="Country Code",
    how="inner"
)

print("country_series_join :", country_series_join.shape)
print("footnote_join       :", footnote_join.shape)
print("data_join           :", data_join.shape)

# 3. Réduction du périmètre des indicateurs

Le dataset contient beaucoup trop d’indicateurs pour une analyse manuelle exhaustive.

Je commence par examiner la colonne `Topic` dans le fichier `Series`, car elle permet de regrouper les indicateurs par grande thématique métier.

In [ ]:
print(series.columns.tolist())
display(series[["Series Code", "Indicator Name", "Topic"]].head(20))

In [ ]:
top_topics = series["Topic"].value_counts(dropna=False).head(20)
display(top_topics.to_frame(name="nb_indicateurs"))

plt.figure(figsize=(12, 6))
plt.bar(top_topics.index.astype(str), top_topics.values)
plt.title("Top 20 des catégories d'indicateurs")
plt.xlabel("Topic")
plt.ylabel("Nombre d'indicateurs")
plt.xticks(rotation=80)
plt.tight_layout()
plt.show()

## Sélection métier des indicateurs

Après revue du dictionnaire `Series` et au regard de la problématique d’expansion, je retiens une première liste de 15 indicateurs couvrant :

- la connectivité numérique ;
- le niveau de développement ;
- l’accès à l’éducation ;
- l’investissement public ;
- la dynamique du marché ;
- la taille potentielle du pays.

In [ ]:
indicateurs_selectionnes = [
    "SP.POP.TOTL",
    "NY.GDP.PCAP.CD",
    "IT.NET.USER.P2",
    "SE.PRM.ENRR",
    "SE.SEC.ENRR",
    "SE.TER.ENRR",
    "SE.XPD.TOTL.GD.ZS",
    "SE.ADT.LITR.ZS",
    "UIS.X.TOTL.FSGOV",
    "SL.UEM.TOTL.ZS",
    "SP.URB.TOTL.IN.ZS",
    "SE.PRM.CMPT.ZS",
    "SE.SEC.CMPT.LO.ZS",
    "SE.TER.CUAT.BA.ZS",
    "IT.CEL.SETS.P2",
]

print("Nombre d'indicateurs sélectionnés :", len(indicateurs_selectionnes))
print(indicateurs_selectionnes)

## Filtrage des tables sur les indicateurs retenus

Je filtre maintenant toutes les tables utiles pour ne garder que ces indicateurs.

Cette étape réduit fortement le périmètre de travail et rend la suite de l’analyse plus lisible.

In [ ]:
footnote_series_col = "SeriesCode" if "SeriesCode" in footnote_clean.columns else "IndicatorCode"

series_filtre = series[series["Series Code"].isin(indicateurs_selectionnes)].copy()
country_series_filtre = country_series_clean[country_series_clean["SeriesCode"].isin(indicateurs_selectionnes)].copy()
footnote_filtre = footnote_clean[footnote_clean[footnote_series_col].isin(indicateurs_selectionnes)].copy()
data_filtre = data_clean[data_clean["Indicator Code"].isin(indicateurs_selectionnes)].copy()

print("series_filtre         :", series_filtre.shape)
print("country_series_filtre :", country_series_filtre.shape)
print("footnote_filtre       :", footnote_filtre.shape)
print("data_filtre           :", data_filtre.shape)

# 4. Réduction du périmètre des années

Le fichier `Data` contient une colonne par année.  
Certaines années futures correspondent à des projections et non à des observations.

Pour une aide à la décision, je privilégie un historique récent et suffisamment renseigné.

In [ ]:
colonnes_annees = [col for col in data_filtre.columns if str(col).isdigit()]
colonnes_annees = sorted(colonnes_annees, key=int)

print("Premières années :", colonnes_annees[:10])
print("Dernières années :", colonnes_annees[-10:])
print("Nombre total de colonnes années :", len(colonnes_annees))

## Taux de valeurs manquantes par année

Avec **Pandas**, la combinaison :

```python
data_filtre[colonnes_annees].isna().mean() * 100
```

fait les opérations suivantes :

- `isna()` transforme les valeurs manquantes en `True` ;
- `mean()` calcule la proportion de `True` par colonne ;
- `* 100` transforme cette proportion en pourcentage.

In [ ]:
taux_nan_par_annee = data_filtre[colonnes_annees].isna().mean().sort_index() * 100
display(taux_nan_par_annee.to_frame(name="pct_valeurs_manquantes"))

plt.figure(figsize=(12, 5))
plt.bar(taux_nan_par_annee.index.astype(str), taux_nan_par_annee.values)
plt.title("Pourcentage de valeurs manquantes par année")
plt.xlabel("Année")
plt.ylabel("% de valeurs manquantes")
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(taux_nan_par_annee.index.astype(str), taux_nan_par_annee.values, marker="o")
plt.title("Taux de valeurs manquantes par année")
plt.xlabel("Année")
plt.ylabel("% de valeurs manquantes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Lecture du graphique :** plus la courbe est haute, plus l’année est pauvre en données.  
Cela aide à justifier le choix de conserver une fenêtre temporelle récente, exploitable et cohérente avec la problématique d’expansion.

Je conserve ici les années **2008 à 2017** :

- elles restent historiques ;
- elles sont plus récentes que le début du dataset ;
- elles permettent de calculer une statistique agrégée par pays et par indicateur.

In [ ]:
annees_retenues = [str(y) for y in range(2008, 2018)]

data_filtre = data_filtre[
    ["Country Name", "Country Code", "Indicator Name", "Indicator Code"] + annees_retenues
].copy()

print("Shape de data_filtre après sélection des années :", data_filtre.shape)

# 5. Richesse des données

Je mesure maintenant la complétude des données :

- par année ;
- par indicateur ;
- par couple indicateur/année.

L’objectif est d’identifier les dimensions les plus exploitables.

## Proportion d’indicateurs renseignés par année

Ici, **Pandas** calcule pour chaque année la part de cellules non manquantes.

Cela permet de repérer les années les plus utilisables.

In [ ]:
proportion_indicateurs_par_annee = data_filtre[annees_retenues].notna().mean()
display(proportion_indicateurs_par_annee.to_frame(name="proportion_renseignee"))

plt.figure(figsize=(10, 5))
plt.plot(proportion_indicateurs_par_annee.index.astype(str), proportion_indicateurs_par_annee.values, marker="o")
plt.title("Proportion d'indicateurs renseignés par année")
plt.xlabel("Année")
plt.ylabel("Proportion renseignée")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Proportion d’années renseignées par indicateur

Ici, j’utilise `groupby("Indicator Code")` pour regrouper les lignes par indicateur, puis une fonction `lambda` pour mesurer la complétude moyenne sur les années retenues.

In [ ]:
proportion_annees_par_indicateur = (
    data_filtre
    .groupby("Indicator Code")[annees_retenues]
    .apply(lambda df: df.notna().mean().mean())
    .sort_values(ascending=False)
)

display(proportion_annees_par_indicateur.to_frame(name="proportion_renseignee"))

plt.figure(figsize=(10, 6))
plt.barh(
    proportion_annees_par_indicateur.index.astype(str),
    proportion_annees_par_indicateur.values
)
plt.title("Proportion moyenne d'années renseignées par indicateur")
plt.xlabel("Proportion renseignée")
plt.ylabel("Indicateur")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## Nombre de pays renseignés par indicateur et par année

Je passe d’abord la table du format large au format long avec `melt()`.

C’est une manipulation très utile en **Pandas** :
- les colonnes années deviennent une colonne `Year` ;
- les valeurs sont rassemblées dans une colonne `Value`.

Cela facilite ensuite les `groupby`.

In [ ]:
data_long = data_filtre.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=annees_retenues,
    var_name="Year",
    value_name="Value"
)

nb_pays_par_indicateur_annee = (
    data_long.dropna(subset=["Value"])
    .groupby(["Indicator Code", "Year"])["Country Code"]
    .nunique()
    .reset_index(name="nb_pays_renseignes")
    .rename(columns={"Indicator Code": "IndicatorCode"})
    .sort_values(["nb_pays_renseignes", "IndicatorCode", "Year"], ascending=[False, True, True])
)

display(nb_pays_par_indicateur_annee.head(20))

top_indicateurs_riches = (
    nb_pays_par_indicateur_annee.groupby("IndicatorCode")["nb_pays_renseignes"]
    .max()
    .sort_values(ascending=False)
    .head(15)
)

display(top_indicateurs_riches.to_frame(name="nb_pays_max_renseignes"))

plt.figure(figsize=(10, 6))
plt.barh(top_indicateurs_riches.index[::-1], top_indicateurs_riches.values[::-1])
plt.title("Indicateurs les plus riches en données")
plt.xlabel("Nombre maximal de pays renseignés")
plt.ylabel("Indicateur")
plt.tight_layout()
plt.show()



## Interprétation de la richesse des indicateurs

Ici, on combine deux logiques :
- une logique **data quality** : garder les indicateurs les mieux renseignés ;
- une logique **métier** : ne pas garder un indicateur uniquement parce qu’il est dense, mais parce qu’il aide réellement à décider où academy pourrait s’implanter.

In [ ]:
df_long = data_filtre.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=annees_retenues,
    var_name="Year",
    value_name="Value"
)

df_pays_par_indicateur_annee = (
    df_long.dropna(subset=["Value"])
    .groupby(["Indicator Code", "Year"])["Country Code"]
    .nunique()
    .reset_index(name="nb_pays_renseignes")
    .sort_values(by="nb_pays_renseignes", ascending=False)
)

display(df_pays_par_indicateur_annee.head(20))

# 6. Agrégation du fichier Data

Le fichier `Data` est à la maille :

**(pays, indicateur, année)**

Je construis maintenant un dataframe plus simple pour l’analyse finale :

- **une ligne = un pays**
- **une colonne = un indicateur**

Pour cela, j’agrège les années par la moyenne avec `pivot_table()`.

In [ ]:
df_final = df_long.pivot_table(
    index=["Country Name", "Country Code"],
    columns="Indicator Code",
    values="Value",
    aggfunc="mean"
).reset_index()

print("Shape de df_final :", df_final.shape)
display(df_final.head())

## Pourquoi utiliser `pivot_table()` ?

Le fichier `Data` est initialement à la maille **pays × indicateur × année**.  
Or, pour calculer une matrice de corrélation puis scorer les pays, il nous faut une table à la maille :

**1 ligne = 1 pays**  
**1 colonne = 1 indicateur**

`pivot_table()` permet justement de **réorganiser** le tableau en agrégeant les années disponibles, ici via une statistique résumée.

# 7. Analyse des corrélations

Je calcule les matrices de corrélation de **Pearson** et de **Spearman** pour repérer les indicateurs redondants.

- **Pearson** mesure surtout les relations linéaires ;
- **Spearman** mesure les relations monotones et est plus robuste à certaines distributions non normales.

In [ ]:
indicateurs_df = df_final.drop(columns=["Country Name", "Country Code"])

corr_pearson = indicateurs_df.corr(method="pearson")
corr_spearman = indicateurs_df.corr(method="spearman")

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(corr_pearson, annot=True, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation - Pearson")
plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_spearman, annot=True, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation - Spearman")
plt.show()

## Pearson vs Spearman

- **Pearson** mesure surtout les relations **linéaires** ;
- **Spearman** mesure plutôt les relations **monotones** à partir des rangs.

Comparer les deux permet de mieux juger si deux indicateurs racontent pratiquement la même chose.  
Si deux colonnes sont très corrélées, elles apportent une information redondante : on peut donc en supprimer une pour simplifier l’analyse.

## Paires trop corrélées

Je liste les paires d’indicateurs dont la corrélation absolue dépasse `0.70`.

Cela permet de décider quels indicateurs conserver ou retirer.

In [ ]:
def lister_paires_correlees(matrice_corr, seuil=0.70):
    colonnes = matrice_corr.columns
    resultats = []

    for i in range(len(colonnes)):
        for j in range(i + 1, len(colonnes)):
            col1 = colonnes[i]
            col2 = colonnes[j]
            corr = matrice_corr.iloc[i, j]

            if pd.notna(corr) and abs(corr) > seuil:
                resultats.append({
                    "indicateur_1": col1,
                    "indicateur_2": col2,
                    "correlation": corr,
                    "correlation_absolue": abs(corr)
                })

    if len(resultats) == 0:
        return pd.DataFrame(columns=["indicateur_1", "indicateur_2", "correlation", "correlation_absolue"])

    return pd.DataFrame(resultats).sort_values(by="correlation_absolue", ascending=False)

seuil_corr = 0.70
paires_pearson = lister_paires_correlees(corr_pearson, seuil_corr)
paires_spearman = lister_paires_correlees(corr_spearman, seuil_corr)

print("Paires trop corrélées selon Pearson :")
display(paires_pearson)

print("Paires trop corrélées selon Spearman :")
display(paires_spearman)

## Sélection finale des indicateurs décorrélés

Après analyse des corrélations et arbitrage métier, je conserve 6 indicateurs :

- `IT.NET.USER.P2`
- `SE.PRM.ENRR`
- `SE.SEC.ENRR`
- `SE.XPD.TOTL.GD.ZS`
- `SL.UEM.TOTL.ZS`
- `SP.POP.TOTL`

Ils couvrent plusieurs dimensions utiles tout en limitant la redondance.

In [ ]:
indicateurs_finaux = [
    "IT.NET.USER.P2",
    "SE.PRM.ENRR",
    "SE.SEC.ENRR",
    "SE.XPD.TOTL.GD.ZS",
    "SL.UEM.TOTL.ZS",
    "SP.POP.TOTL",
]

df_final_decorrele = df_final[["Country Name", "Country Code"] + indicateurs_finaux].copy()

print("Shape avant :", df_final.shape)
print("Shape après :", df_final_decorrele.shape)
print("Indicateurs gardés :")
print(indicateurs_finaux)

display(df_final_decorrele.head())

# 8. Analyse univariée

Je calcule maintenant les statistiques descriptives de chaque indicateur et j’observe leur distribution.

La fonction ci-dessous combine :
- `describe()` pour les statistiques descriptives ;
- un histogramme Matplotlib ;
- un boxplot Matplotlib.

Le couple histogramme + boxplot permet de voir :
- la forme de la distribution ;
- l’étendue des valeurs ;
- la présence éventuelle d’outliers.

In [ ]:
def analyser_indicateur(df, colonne):
    serie = df[colonne].dropna()

    print("=" * 70)
    print(f"Indicateur : {colonne}")
    print("=" * 70)
    display(serie.describe().to_frame(name=colonne))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(serie, bins=20)
    axes[0].set_title(f"Histogramme - {colonne}")
    axes[0].set_xlabel(colonne)
    axes[0].set_ylabel("Nombre de pays")

    axes[1].boxplot(serie, vert=False)
    axes[1].set_title(f"Boxplot - {colonne}")
    axes[1].set_xlabel(colonne)

    plt.tight_layout()
    plt.show()

In [ ]:
for col in indicateurs_finaux:
    analyser_indicateur(df_final_decorrele, col)

## Comment lire `describe()` pour un indicateur ?

La sortie de `describe()` donne notamment :
- `count` : nombre de valeurs disponibles ;
- `mean` : moyenne ;
- `std` : dispersion ;
- `min` / `max` : bornes observées ;
- `25%`, `50%`, `75%` : quantiles utiles pour comprendre la distribution.

L’histogramme complète cette lecture en montrant la forme globale de la distribution, tandis que le boxplot aide à repérer rapidement les valeurs extrêmes.

## Interprétation synthétique des distributions

- `IT.NET.USER.P2` distingue bien les pays très connectés des pays moins avancés numériquement.
- `SE.PRM.ENRR` est souvent élevé, donc moins discriminant seul.
- `SE.SEC.ENRR` apporte davantage de variation entre pays.
- `SE.XPD.TOTL.GD.ZS` mesure l’effort public consacré à l’éducation.
- `SL.UEM.TOTL.ZS` est un indicateur défavorable : plus il est élevé, moins l’environnement paraît attractif.
- `SP.POP.TOTL` permet d’intégrer la taille potentielle du marché, mais sa distribution est très asymétrique.

# 9. Méthode quantitative de sélection des pays

Je construis un **score composite** en quatre étapes :

1. normalisation min-max des indicateurs ;
2. inversion du chômage ;
3. pondération des dimensions ;
4. classement des pays.

## Pourquoi NumPy ici ?

`NumPy` permet de vectoriser les calculs, c’est-à-dire de traiter des colonnes entières très efficacement sans boucle complexe.  
Je l’utilise ici pour sécuriser la normalisation et le calcul du score.

In [ ]:
df_score = df_final_decorrele.copy()

# Normalisation min-max : chaque indicateur est ramené entre 0 et 1
for col in indicateurs_finaux:
    min_val = df_score[col].min()
    max_val = df_score[col].max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        df_score[col + "_norm"] = np.nan
    else:
        df_score[col + "_norm"] = (df_score[col] - min_val) / (max_val - min_val)

# Le chômage est défavorable : plus il est faible, mieux c'est
df_score["SL.UEM.TOTL.ZS_norm"] = 1 - df_score["SL.UEM.TOTL.ZS_norm"]

poids = {
    "IT.NET.USER.P2_norm": 0.25,
    "SE.PRM.ENRR_norm": 0.10,
    "SE.SEC.ENRR_norm": 0.20,
    "SE.XPD.TOTL.GD.ZS_norm": 0.15,
    "SL.UEM.TOTL.ZS_norm": 0.15,
    "SP.POP.TOTL_norm": 0.15,
}

colonnes_score = list(poids.keys())
vecteur_poids = np.array([poids[col] for col in colonnes_score])

# Remplissage temporaire à 0 pour calculer le score sans erreur,
# puis division par la somme des poids effectivement disponibles
matrice = df_score[colonnes_score].fillna(0).to_numpy()
masque_disponible = df_score[colonnes_score].notna().to_numpy().astype(float)

numerateur = matrice @ vecteur_poids
denominateur = masque_disponible @ vecteur_poids

df_score["score_pays"] = np.where(denominateur > 0, numerateur / denominateur, np.nan)
df_score = df_score.sort_values(by="score_pays", ascending=False)

top_pays = df_score[["Country Name", "Country Code", "score_pays"]].head(10).copy()
display(top_pays)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_pays["Country Name"][::-1], top_pays["score_pays"][::-1])
plt.title("Top 10 des pays recommandés")
plt.xlabel("Score composite")
plt.ylabel("Pays")
plt.tight_layout()
plt.show()

# 10. Liste finale des pays favoris

L’affichage ci-dessous constitue la **liste finale des pays favoris / recommandés** à partir du score composite construit sur les indicateurs retenus.

In [ ]:
top_pays_final = top_pays.reset_index(drop=True).copy()
top_pays_final.index = top_pays_final.index + 1
display(top_pays_final)

## Recommandation managériale

La liste affichée ci-dessus n’est pas seulement un résultat technique : c’est une **présélection de pays cibles** pour alimenter la réflexion stratégique de Mark.

Elle peut servir de base à une discussion avec des critères complémentaires :
- taille de marché adressable ;
- concurrence locale ;
- réglementation de l’éducation / du numérique ;
- coût d’entrée commercial ;
- compatibilité avec l’offre lycée / université de academy.

# 11. Conclusion

Les pays qui ressortent le plus sont majoritairement des pays :

- très connectés ;
- stables ;
- bien structurés sur le plan éducatif ;
- portés par de bons niveaux de développement.

Cette analyse constitue une **base de présélection** utile pour une décision d’expansion, mais elle doit être complétée par d’autres critères business :

- concurrence ;
- réglementation ;
- coût d’entrée ;
- stratégie commerciale propre à l’entreprise.

# 12. Export des résultats

Je sauvegarde :

- le dataframe final décorrélé ;
- la liste finale des pays recommandés ;
- un export simplifié avec uniquement les noms des pays favoris.


In [ ]:
df_final_decorrele.to_csv(base_path / "df_final_decorrele.csv", index=False)
top_pays_final.to_csv(base_path / "top_pays_recommandes.csv", index=False)

# Export simplifié : uniquement la liste des pays favoris
col_pays = None
for candidate in ["Country Name", "country", "Country", "pays", "Pays"]:
    if candidate in top_pays_final.columns:
        col_pays = candidate
        break

if col_pays is None:
    raise KeyError("Aucune colonne de pays trouvée dans top_pays_final.")

liste_pays_favoris = top_pays_final[[col_pays]].copy()
liste_pays_favoris.columns = ["Pays favori"]
liste_pays_favoris.to_csv(base_path / "liste_pays_favoris.csv", index=False)

print("Fichiers exportés :")
print("-", base_path / "df_final_decorrele.csv")
print("-", base_path / "top_pays_recommandes.csv")
print("-", base_path / "liste_pays_favoris.csv")

display(liste_pays_favoris)


## Conseils pour la soutenance / la restitution

Pour une présentation orale ou des slides, vous pouvez structurer votre discours ainsi :

1. **Contexte** : academy cherche des pays à fort potentiel ;
2. **Données disponibles** : cinq fichiers Banque mondiale, pas tous directement exploitables ;
3. **Nettoyage** : suppression des doublons, colonnes vides, faux pays ;
4. **Réduction du périmètre** : sélection d’indicateurs et d’années pertinentes ;
5. **Agrégation** : passage à une table pays × indicateurs ;
6. **Analyse statistique** : corrélations, distributions, score composite ;
7. **Conclusion** : liste priorisée de pays à étudier en priorité.

L’idée n’est pas de réciter du code, mais de montrer une démarche claire, argumentée et reproductible.

# 13. Affichage final de la liste des pays favoris

Pour éviter toute ambiguïté, je réaffiche ici **tout à la fin du notebook** la liste finale des pays recommandés.



In [ ]:
# Réaffichage final demandé : la liste des pays favoris apparaît explicitement en fin de notebook
if "liste_pays_favoris" not in globals():
    col_pays = None
    for candidate in ["Country Name", "country", "Country", "pays", "Pays"]:
        if candidate in top_pays_final.columns:
            col_pays = candidate
            break
    if col_pays is None:
        raise KeyError("Aucune colonne de pays trouvée dans top_pays_final.")
    liste_pays_favoris = top_pays_final[[col_pays]].copy()
    liste_pays_favoris.columns = ["Pays favori"]

liste_pays_favoris_finale = liste_pays_favoris.reset_index(drop=True).copy()
liste_pays_favoris_finale.index = liste_pays_favoris_finale.index + 1
print("Liste finale des pays favoris :")
display(liste_pays_favoris_finale)

